# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Title:** Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells
- **Identifier:** 10.71728/senscience.vq4a-28xa
- **Source:** [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

`mlcroissant` enables programmatic inspection of the dataset schema. We will enumerate all record sets in the metadata, then for each, display its fields and columns referenced by their `@id`.

> Note: All entity references use their `@id` for consistency.

In [ ]:
# Display all record sets and their fields and columns by @id

def safe_get(obj, attr, default='N/A'):
    try:
        return getattr(obj, attr)
    except Exception:
        return default

record_set_ids = []

print("Available Record Sets:")
for record_set in metadata.record_sets:
    rs_id = getattr(record_set, '@id', None)
    record_set_ids.append(rs_id)
    print(f"- RecordSet @id: {rs_id}")
    print(f"  Name: {safe_get(record_set, 'name')}")
    print(f"  Description: {safe_get(record_set, 'description')}")
    # List fields
    print("  Fields:")
    for field in getattr(record_set, 'fields', []):
        print(f"    - Field @id: {getattr(field, '@id', 'N/A')}, name: {safe_get(field, 'name')}, dtype: {safe_get(field, 'data_type')}")
    # List columns & sources
    print("  Columns:")
    for column in getattr(record_set, 'columns', []):
        col_id = getattr(column, '@id', 'N/A')
        print(f"    - Column @id: {col_id}, source: {safe_get(column, 'source')}, extract: {safe_get(column, 'extract')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

> Here we demonstrate loading **one** record set. Update the variable `record_set_to_load` with the desired `@id` as revealed above (or load all record sets as needed).

We'll show both the first few records and the columns loaded.

In [ ]:
# Extract data from one or more record sets

# Choose the first record set as example
if len(record_set_ids) == 0:
    raise RuntimeError("No record sets found in the dataset metadata.")

record_set_to_load = record_set_ids[0]

records = list(dataset.records(record_set=record_set_to_load))
df = pd.DataFrame(records)
print(f"Loaded columns for record set {record_set_to_load}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)

Apply basic data processing steps such as filtering records based on field values, normalizing numeric fields, and aggregating by key variables.


In [ ]:
# Identify a numeric field for analysis. Here, we attempt to auto-detect one.

import numpy as np

# Attempt to find a numeric column
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break

if numeric_field is None:
    # Try to coerce columns to numeric to auto-detect
    for col in df.columns:
        # Try convert to float; skip if error
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except Exception:
            pass
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

if numeric_field is None:
    raise ValueError("No numeric field found for EDA!")

print(f"Using numeric field for analysis: {numeric_field}")

# Filter records (e.g. values > threshold)
threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {round(threshold,2)}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt to group by a categorical field if present
potential_group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
if len(potential_group_fields) > 0:
    group_field = potential_group_fields[0]
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())
else:
    print("No categorical group field found to group by.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the selected numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If we have a categorical field, create a boxplot
if 'group_field' in locals() and group_field in df.columns:
    plt.figure(figsize=(12,5))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- This notebook demonstrated how to load, inspect, and analyze a Croissant dataset using record set, field, and column `@id` references with `mlcroissant`.
- Common EDA and basic visualization are provided as a starting point for further domain analysis.